## Environment setup

In [ ]:
from datetime import datetime 
import importlib
from functools import partial

import tensorflow as tf
from tensorflow.data import Dataset, TFRecordDataset
import tensorflow_datasets as tfds

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns

import sys
sys.path.append("/home/akalinow/scratch/ELITPC/TPCReco/PythonAnalysis/python/")

#Increase plots font size
params = {'legend.fontsize': 'xx-large',
          'figure.figsize': (10, 7),
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

### Load dataset and convert to pandas DataFrame
```python

In [ ]:
%%time 

dataset = tf.data.Dataset.load('MergedEvent_Track3D_TwoProng_gun_MC', compression="GZIP")

columns = ["xVtx", "xAlpha", "xCarbon", "yVtx", "yAlpha", "yCarbon", "zVtx", "zAlpha", "zCarbon"]

data = dataset.map(lambda x: tf.reshape(x["sim"][1], (1,-1))).unbatch().as_numpy_iterator()
df = pd.DataFrame(data, columns=columns)

data = dataset.map(lambda x: tf.reshape(x["reco"][1], (1,-1))).unbatch().as_numpy_iterator()
df_reco = pd.DataFrame(data, columns=columns)

df = df.merge(df_reco, left_index=True, right_index=True, suffixes=("_sim", "_reco"), how="outer");

### Resolution plots

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

#plf.controlPlots(df)
#plf.plotEndPointRes(df=df, edge="Vtx")
#plf.plotEndPointRes(df=df, edge="Alpha")
#plf.plotEndPointRes(df=df, edge="Carbon")


#plf.plotLengthPull(df, partName="Alpha")
#plf.plotLengthPull(df, partName="Carbon")
#plf.plotLengthPullEvolution(df)
plf.plotOpeningAngleCos(df)

'''
plf.controlPlots(df)
plf.plotEndPointRes(df=df, edge="Start", partIdx=1)
plf.plotEndPointRes(df=df, edge="Stop", partIdx=1)

plf.plotEndPointRes(df=df, edge="Start", partIdx=2)
plf.plotEndPointRes(df=df, edge="Stop", partIdx=2)

plf.plotLengthPull(df, partIdx=1)
plf.plotLengthPull(df, partIdx=2)
plf.plotLengthPullEvolution(df)
plf.plotOpeningAngleCos(df)
'''


### Resolution plots for filtered dataset

In [ ]:
mask = np.abs(df["GEN_StartPosX"] - df["RECO_StartPosX"])<1
df_filtered = df[mask]

mask = np.abs(df_filtered["GEN_StartPosY"] - df_filtered["RECO_StartPosY"])<1
df_filtered = df_filtered[mask]

mask = np.abs(df_filtered["GEN_StopPosX_Part1"] - df_filtered["RECO_StopPosX_Part1"])<10
df_filtered = df_filtered[mask]

print(df_filtered.describe())

plf.plotEndPointRes(df=df_filtered, edge="Start", partIdx=1)
plf.plotEndPointRes(df=df_filtered, edge="Stop", partIdx=1)

plf.plotEndPointRes(df=df_filtered, edge="Start", partIdx=2)
plf.plotEndPointRes(df=df_filtered, edge="Stop", partIdx=2)

plf.plotLengthPull(df_filtered, partIdx=1)
plf.plotLengthPull(df_filtered, partIdx=2)
plf.plotLengthPullEvolution(df_filtered)
plf.plotOpeningAngleCos(df_filtered)

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

plf.plotLengthPullEvolution(df_filtered)

## 2D plots 

In [ ]:
params = {'legend.fontsize': 'xx-large',
          'figure.figsize': (14, 10),
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large',
         #'xticks':'major_ticks_top'
         }

plt.rcParams.update(params)


fig, axes = plt.subplots(2, 2, figsize = (10, 10))

'''
trackTree->Draw("(alphaRangeReco-alphaRangeGen)/alphaRangeGen:alphaRangeGen>>hRangeResVsRangeGen(21,0, 70,  41,-0.5,0.5)","","goff");
trackTree->Draw("(alphaRangeReco-alphaRangeGen)/alphaRangeGen:cosThetaGen>>hRangeResVsCosTheta(21,-1, 1,  41,-0.5,0.5)","","goff");
trackTree->Draw("abs(cosThetaReco)-abs(cosThetaGen):cosThetaGen>>hCosThetaResVsCosTheta(21,-1, 1,  41,-0.5,0.5)","","goff");
trackTree->Draw("asin(sin(phiReco-phiGen)):cosThetaGen>>hPhiResVsCosTheta(21,-1, 1,  41,-0.5,0.5)","","goff");
trackTree->Draw("(chargeReco-chargeGen)/chargeGen:cosThetaGen>>hChargeResVsCosTheta(21,-1, 1,  41,-0.5,0.5)","","goff");
'''

x = df["alphaRangeGen"]
y = df.eval("(alphaRangeReco -alphaRangeGen)/alphaRangeGen")
xBins = np.linspace(0,60,20)
yBins = np.linspace(-0.5,0.5,20)
axes[0,0].hist2d(x, y, bins=(xBins, yBins), cmin=10, label="length")
axes[0,0].set_xlabel(r'$\alpha~range [mm]$')
axes[0,0].set_ylabel(r'$\frac{RECO-GEN}{GEN}$')

x = df["cosThetaGen"]
xBins = np.linspace(-1,1,20)
yBins = np.linspace(-0.5,0.5,20)
axes[0,1].hist2d(x, y, bins=(xBins, yBins), cmin=10, label="length")
axes[0,1].set_xlabel(r'$\cos(\theta)$')
axes[0,1].set_ylabel(r'$\frac{RECO-GEN}{GEN}$')

x = df["phiGen"]
xBins = np.linspace(-np.pi,np.pi,20)
yBins = np.linspace(-0.5,0.5,20)
axes[1,0].hist2d(x, y, bins=(xBins, yBins), cmin=10, label="length")
axes[1,0].set_xlabel(r'$\varphi$')
axes[1,0].set_ylabel(r'$\frac{RECO-GEN}{GEN}$')

x = df["dEdxFitChi2"]
xBins = np.linspace(x.min(),x.median(),20)
yBins = np.linspace(-0.5,0.5,20)
axes[1,1].hist2d(x, y, bins=(xBins, yBins), cmin=10, label="length")
axes[1,1].set_xlabel(r'$\frac{dE}{dx}$ fit loss func.')
axes[1,0].set_ylabel(r'$\frac{RECO-GEN}{GEN}$')

plt.subplots_adjust(bottom=0.15, left=0.05, right=0.95, wspace=0.35, hspace=0.3)
#plt.savefig("fig_png/2D_plots.png", bbox_inches="tight")